# DIMER posterior-predictive video

Real-vs-synthetic viewer for the posterior-predictive check. The engine
`Script_Bank/Analysis/SRM_AND_SBI_DIMER_ALP_DETECTOR_Posterior_Predictive_Video.py` renders a
synthetic video from a real recording's MAP imaging estimate and persists it
(`*_Synthetic_Video.npz`); this notebook only **views** the persisted clip. It never runs a
simulation, so scrubbing, zooming, and re-running cells never regenerate the trajectory. To get
a new trajectory, re-run the engine (a deliberate step), optionally with `--seed` for
reproducibility.

**How to run.** This is a pure viewer -- it needs only `numpy`, `matplotlib`, and `ipywidgets`
(no project package, no `MACHINE_PROFILE`, no ReaDDy). Render a clip with the engine on the
machine that holds the data (e.g. rcl01), copy the resulting `*_Synthetic_Video.npz` to this
machine, set `CLIP_PATH` in the next code cell, and launch `jupyter lab`. The clip is
self-contained (real + synthetic + provenance), so it opens anywhere.

The synthetic's **motion** is a fresh RDS-nuisance draw (not the real track); the comparison
reads **imaging appearance** -- PSF, brightness, noise, flicker. The clip is kept at **16-bit**;
only the display color scaling is normalized, and it is selectable via `NORM_MODE` (next cell).
The estimator itself was calibrated on the fixed 8-bit rescale.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, FloatSlider, interact
# Pure viewer: needs only numpy + matplotlib + ipywidgets. No project package, no
# MACHINE_PROFILE, no ReaDDy -- rendering happens in the engine on the data machine.

In [ ]:
# Point CLIP_PATH at a *_Synthetic_Video.npz produced by the engine
# (SRM_AND_SBI_DIMER_ALP_DETECTOR_Posterior_Predictive_Video.py). Copy it to this machine
# first; it is self-contained (real + synthetic + provenance), so any absolute path works.
CLIP_PATH = "<absolute path to ..._Synthetic_Video.npz>"

# Display normalization. The data stay 16-bit; this only sets how the magma colormap is scaled:
#   "autoscale"  -> vmin = vmax = None: magma stretched to each image's own min/max
#                   (the reference viewer's behavior; per-frame in scrub, frame-0 in playback).
#   "percentile" -> a fixed per-video window [p0.5, p99.5], stable across all frames.
NORM_MODE = "autoscale"

d = np.load(CLIP_PATH, allow_pickle=True)
real, synth = d["real"], d["synth"]            # (n_frames, H, W) uint16
n_frames = int(d["n_frames"])
fps = round(1.0 / float(d["frame_time_seconds"]))
print(f"real {real.shape}  synth {synth.shape}  |  {fps} fps  |  kind={str(d['kind'])} "
      f"cell={int(d['cell'])}  MAP source={str(d['map_source'])}  seed={int(d['seed'])}")
print(f"ADU  real  min {int(real.min())} median {int(np.median(real))} max {int(real.max())}")
print(f"ADU  synth min {int(synth.min())} median {int(np.median(synth))} max {int(synth.max())}")

# Per-video (global) percentile windows -- computed once so scrubbing/playback do not recompute.
_PCTL = {"real": (float(np.percentile(real, 0.5)), float(np.percentile(real, 99.5))),
         "synth": (float(np.percentile(synth, 0.5)), float(np.percentile(synth, 99.5)))}

def clim(which):
    """(vmin, vmax) for a panel ('real'/'synth') under NORM_MODE."""
    return _PCTL[which] if NORM_MODE == "percentile" else (None, None)

print(f"NORM_MODE={NORM_MODE}  " + (f"[p0.5, p99.5]  real {tuple(round(v) for v in _PCTL['real'])}  "
      f"synth {tuple(round(v) for v in _PCTL['synth'])}" if NORM_MODE == "percentile"
      else "(magma autoscaled to each image's own min/max)"))

In [ ]:
# Scrub frames and zoom a shared region-of-interest into BOTH panels together.
# zoom = 1 shows the full frame; higher zoom crops tighter around (center x, center y).
H, W = real.shape[1], real.shape[2]

def _roi(frame, cx, cy, zoom):
    half_y, half_x = int(H / (2 * zoom)), int(W / (2 * zoom))
    y0, y1 = max(0, cy - half_y), min(H, cy + half_y)
    x0, x1 = max(0, cx - half_x), min(W, cx + half_x)
    return frame[y0:y1, x0:x1], (x0, x1, y0, y1)

def show(frame, cx, cy, zoom):
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    for a, arr, title, which in ((ax[0], real, "REAL", "real"),
                                 (ax[1], synth, "SYNTH (MAP)", "synth")):
        crop, ext = _roi(arr[frame], cx, cy, zoom)
        vmin, vmax = clim(which)
        a.imshow(crop, cmap="magma", origin="lower", interpolation="none",
                 vmin=vmin, vmax=vmax, extent=ext)
        a.set_title(f"{title}   frame {frame}/{n_frames - 1}   zoom x{zoom:g}")
    plt.tight_layout(); plt.show()

interact(
    show,
    frame=IntSlider(min=0, max=n_frames - 1, step=1, value=n_frames // 2, description="frame"),
    cx=IntSlider(min=0, max=W - 1, step=1, value=W // 2, description="center x"),
    cy=IntSlider(min=0, max=H - 1, step=1, value=H // 2, description="center y"),
    zoom=FloatSlider(min=1.0, max=8.0, step=0.5, value=1.0, description="zoom"),
);

## Playback (real time)

Play both clips side by side at the native frame rate. The player embeds every frame, so a long
clip builds a large widget -- the embed limit is raised below so no frames are dropped. Increase
`PLAY_EVERY` to lighten/speed the build (it stays real-time). The scrub cell above already gives
frame-accurate access to every frame.

In [ ]:
import matplotlib as mpl
from matplotlib import animation
from IPython.display import HTML

mpl.rcParams["animation.embed_limit"] = 256   # MB; a 1000-frame 2-panel player exceeds the 20 MB default
PLAY_EVERY = 1                                 # stride: raise (2, 5, ...) to shrink the player; stays real-time

idx = list(range(0, n_frames, PLAY_EVERY))
fig, ax = plt.subplots(1, 2, figsize=(9, 4.6))
ims = []
for a, arr, title, which in ((ax[0], real, "REAL", "real"), (ax[1], synth, "SYNTH (MAP)", "synth")):
    vmin, vmax = clim(which)
    im = a.imshow(arr[0], cmap="magma", origin="lower", interpolation="none", vmin=vmin, vmax=vmax)
    a.set_title(title); a.set_xticks([]); a.set_yticks([]); ims.append(im)
plt.close(fig)   # suppress the static duplicate; the player below renders it

def _update(i):
    ims[0].set_data(real[i]); ims[1].set_data(synth[i])
    fig.suptitle(f"frame {i}/{n_frames - 1}   ({fps} fps)")
    return ims

anim = animation.FuncAnimation(fig, _update, frames=idx, interval=1000.0 * PLAY_EVERY / fps, blit=False)
HTML(anim.to_jshtml())